## Libraries

In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDRegressor
from scipy.stats import norm
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.neighbors import NearestNeighbors
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm

## Config

In [2]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_END_DATE  = pd.Timestamp("2022-03-31")
TEST_START_DATE = pd.Timestamp("2022-04-01")

# ---- INSERT YOUR BEST LAG SET & PARAMS HERE ----
# These should come from your tuning results_df.iloc[0]["params"] + lag_set
best_lag_set = [1, 2, 3, 12]  # example: replace with your selected lag set

best_kernel_params = {
    "temporal_gammas": (0.05,0.1),
    "temporal_n_components": 150,
    "spatial_gamma": 0.05,
    "spatial_n_components": 150,
}

best_sgd_params = {
    "alpha": 1e-6,
    "epsilon": 0.1,
    # keep solver knobs aligned to tuning code
    "learning_rate": "invscaling",
    "eta0": 0.01,
    "max_iter": 3000,
    "tol": 1e-3,
    "random_state": 42,
}

# ---- FEATURES (must match your dataset column names) ----
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4",
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

# Block definitions (same as tuning)
spatial_cols = ["area_km2", "centroid_x", "centroid_y", "CoL_distance_km"]
other_continuous_cols = [c for c in continuous_cols if c not in spatial_cols]
other_cols = other_continuous_cols + categorical_cols


## Metric functions

In [3]:
def mae(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.mean(np.abs(y - yhat))

def rmse(y, yhat):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return np.sqrt(np.mean((y - yhat) ** 2))

def smape(y, yhat, eps=1e-8):
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    return 100.0 * np.mean(2.0 * np.abs(yhat - y) / (np.abs(y) + np.abs(yhat) + eps))

def mase(y, yhat, y_train, m=12, eps=1e-8):
    """Global MASE using in-sample seasonal naive with period m on y_train."""
    y = np.asarray(y)
    yhat = np.asarray(yhat)
    y_train = np.asarray(y_train)
    if len(y_train) <= m:
        return np.nan
    naive_diff = np.abs(y_train[m:] - y_train[:-m])
    scale = np.mean(naive_diff) + eps
    return np.mean(np.abs(y - yhat)) / scale

def directional_accuracy(df, entity_col, time_col, target_col, pred_col):
    """Fraction of times sign of month-on-month change is correct."""
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_diff"] = df.groupby(entity_col)[target_col].diff()
    df["yhat_diff"] = df.groupby(entity_col)[pred_col].diff()
    mask = df["y_diff"].notna() & df["yhat_diff"].notna()
    same_dir = np.sign(df.loc[mask, "y_diff"]) == np.sign(df.loc[mask, "yhat_diff"])
    return same_dir.mean()

def growth_rate_error(df, entity_col, time_col, target_col, pred_col, m=12):
    """
    12-month growth rate error:
    g_t = (y_t - y_{t-m}) / y_{t-m}
    Returns MAE of growth-rate error.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["y_lag_m"] = df.groupby(entity_col)[target_col].shift(m)
    df["yhat_lag_m"] = df.groupby(entity_col)[pred_col].shift(m)

    mask = df["y_lag_m"].notna() & df["yhat_lag_m"].notna() & (df["y_lag_m"] != 0)
    y_gr = (df.loc[mask, target_col] - df.loc[mask, "y_lag_m"]) / df.loc[mask, "y_lag_m"]
    yhat_gr = (df.loc[mask, pred_col] - df.loc[mask, "yhat_lag_m"]) / df.loc[mask, "yhat_lag_m"]

    return np.mean(np.abs(y_gr - yhat_gr))

def morans_i(
    residuals,
    xs,
    ys,
    k=5,
    eps=1e-8,
    symmetric=True,
    row_standardize=True,
    permutations=0,
    random_state=None,
):
    """
    Compute Moran's I for residuals using k-nearest neighbours
    with inverse-distance weights.
    """
    residuals = np.asarray(residuals, dtype=float)
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)

    N = len(residuals)
    if not (len(xs) == len(ys) == N):
        raise ValueError("residuals, xs, ys must all have the same length")

    # Center residuals
    x_mean = residuals.mean()
    x_dev = residuals - x_mean

    # Build kNN graph
    coords = np.column_stack([xs, ys])
    nbrs = NearestNeighbors(n_neighbors=k + 1).fit(coords)
    distances, indices = nbrs.kneighbors(coords)

    # Weight matrix W (dense; for large N you might switch to sparse)
    W = np.zeros((N, N), dtype=float)
    for i in range(N):
        neigh_idx = indices[i, 1:]          # skip self at index 0
        w = 1.0 / (distances[i, 1:] + eps)  # inverse-distance weights
        W[i, neigh_idx] = w

    # Optional symmetrisation
    if symmetric:
        W = 0.5 * (W + W.T)

    # Optional row standardisation
    if row_standardize:
        row_sums = W.sum(axis=1, keepdims=True)
        W = np.where(row_sums > 0, W / (row_sums + eps), 0.0)

    S0 = W.sum()

    # Moran's I numerator and denominator (vectorised)
    num = (W * (x_dev[:, None] * x_dev[None, :])).sum()
    den = (x_dev ** 2).sum() + eps

    I_obs = (N / S0) * (num / den)

    result = {
        "I": I_obs,
        "S0": S0,
        "permutations": None,
        "z_score": None,
        "p_value": None,
    }

    # Optional permutation test
    if permutations > 0:
        if isinstance(random_state, np.random.Generator):
            rng = random_state
        else:
            rng = np.random.default_rng(random_state)

        perm_I = np.empty(permutations, dtype=float)
        for b in range(permutations):
            perm = rng.permutation(x_dev)
            num_b = (W * (perm[:, None] * perm[None, :])).sum()
            perm_I[b] = (N / S0) * (num_b / den)

        mean_perm = perm_I.mean()
        std_perm = perm_I.std(ddof=1) + eps
        z = (I_obs - mean_perm) / std_perm

        extreme = np.sum(np.abs(perm_I - mean_perm) >= np.abs(I_obs - mean_perm))
        p_val = (extreme + 1) / (permutations + 1)

        result.update(
            {
                "permutations": perm_I,
                "z_score": z,
                "p_value": p_val,
            }
        )

    return result

def crps_gaussian(y,mu,sigma,eps=1e-8):
    a=(y-mu)/(sigma+eps)
    return np.mean(sigma*(1/np.sqrt(np.pi)-2*norm.pdf(a)-a*(2*norm.cdf(a)-1)))

## Multi kernel

In [4]:
class MultiRBFFeatures(BaseEstimator, TransformerMixin):
    """Concatenate multiple RBFSampler maps (multi-scale RBF)."""
    def __init__(self, gammas=(0.05, 0.1, 0.2), n_components=200, random_state=42):
        self.gammas = tuple(gammas)
        self.n_components = int(n_components)
        self.random_state = int(random_state)

    def fit(self, X, y=None):
        self.samplers_ = []
        for i, g in enumerate(self.gammas):
            s = RBFSampler(
                gamma=float(g),
                n_components=self.n_components,
                random_state=self.random_state + i
            )
            s.fit(X)
            self.samplers_.append(s)
        return self

    def transform(self, X):
        return np.hstack([s.transform(X) for s in self.samplers_])


## Rolling STL feature builder

In [5]:
def add_rolling_stl_components(
    df,
    entity_col,
    time_col,
    target_col,
    period=12,
    min_history=24,
    window=120,        # set None for expanding, or e.g. 120 to match your 10y window
    robust=True,
    show_progress=True,
):
    """
    For each LA, compute STL components at time t using only y up to time t.
    We assign the *last* STL values from the fitted history to that time t.

    IMPORTANT:
    - This creates stl_trend/stl_seasonal/stl_resid for each row.
    - You should only use *lags* of these components (e.g., lag1/lag12/lag24)
      when predicting y_t, otherwise you'd leak y_t into its own features.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["stl_trend"] = np.nan
    df["stl_seasonal"] = np.nan
    df["stl_resid"] = np.nan

    # Precompute total iterations for tqdm
    groups = list(df.groupby(entity_col))
    total_steps = sum(len(sub) for _, sub in groups)

    iterator = tqdm(
        groups,
        desc="Rolling STL per LA",
        total=len(groups),
        leave=True,
        disable=not show_progress,
    )

    for la, sub in iterator:
        sub = sub.sort_values(time_col)
        y = sub[target_col].astype(float).values
        n = len(sub)

        for t in range(n):
            start = 0 if window is None else max(0, t - window + 1)
            hist = y[start : t + 1]

            if len(hist) < min_history or np.isnan(hist).any():
                continue

            try:
                res = STL(hist, period=period, robust=robust).fit()

                idx = sub.index[t]
                df.loc[idx, "stl_trend"]    = res.trend[-1]
                df.loc[idx, "stl_seasonal"] = res.seasonal[-1]
                df.loc[idx, "stl_resid"]    = res.resid[-1]

            except Exception:
                continue

    return df

## Load data

In [6]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.query('Date < "2024-03-31"')

df = df.sort_values([ENTITY_COL,TIME_COL]).reset_index(drop=True)

df_train_all = df[df[TIME_COL] <= TRAIN_END_DATE].copy()
df_test      = df[df[TIME_COL] >= TEST_START_DATE].copy()

## Training

In [ ]:
combined = pd.concat([df_train_all, df_test], axis=0).sort_values([ENTITY_COL, TIME_COL]).copy()
combined = add_rolling_stl_components(
    combined,
    entity_col=ENTITY_COL,
    time_col=TIME_COL,
    target_col=TARGET_COL,
    period=12,
    min_history=24,
    robust=True,
)

In [11]:


# Create selected lag columns
for lag in best_lag_set:
    for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
        combined[f"{comp}_lag{lag}"] = combined.groupby(ENTITY_COL)[comp].shift(lag)

lag_cols = [f"{c}_lag{l}" for c in ["stl_trend", "stl_seasonal", "stl_resid"] for l in best_lag_set]
feature_cols = continuous_cols + categorical_cols + lag_cols

df_train = combined[combined[TIME_COL] <= TRAIN_END_DATE].dropna(subset=feature_cols).copy()
df_test  = combined[combined[TIME_COL] >= TEST_START_DATE].dropna(subset=feature_cols).copy()

# =========================================================
# Build block matrices
# =========================================================
y_train_raw = df_train[TARGET_COL].values.reshape(-1, 1)
y_test_true = df_test[TARGET_COL].values

# Temporal block = STL lags only
X_train_temp = df_train[lag_cols].copy()
X_test_temp  = df_test[lag_cols].copy()

# Spatial block
X_train_spat = df_train[spatial_cols].copy()
X_test_spat  = df_test[spatial_cols].copy()

# Other block = remaining continuous + categoricals (linear)
X_train_other = df_train[other_cols].copy()
X_test_other  = df_test[other_cols].copy()

# =========================================================
# Scale blocks (train only)
# =========================================================
temp_scaler = StandardScaler()
X_train_temp = temp_scaler.fit_transform(X_train_temp)
X_test_temp  = temp_scaler.transform(X_test_temp)

spat_scaler = StandardScaler()
X_train_spat = spat_scaler.fit_transform(X_train_spat)
X_test_spat  = spat_scaler.transform(X_test_spat)

other_scaler = StandardScaler()
X_train_other_scaled = X_train_other.copy()
X_test_other_scaled  = X_test_other.copy()

X_train_other_scaled[other_continuous_cols] = other_scaler.fit_transform(
    X_train_other_scaled[other_continuous_cols]
)
X_test_other_scaled[other_continuous_cols] = other_scaler.transform(
    X_test_other_scaled[other_continuous_cols]
)

Z_train_other = X_train_other_scaled.values
Z_test_other  = X_test_other_scaled.values

# Scale y (train only)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train_raw).ravel()

# =========================================================
# Kernel maps (FINAL selected)
# =========================================================
temporal_map = MultiRBFFeatures(
    gammas=best_kernel_params["temporal_gammas"],
    n_components=best_kernel_params["temporal_n_components"],
    random_state=42,
)
Z_train_temp = temporal_map.fit_transform(X_train_temp)
Z_test_temp  = temporal_map.transform(X_test_temp)

spatial_map = RBFSampler(
    gamma=best_kernel_params["spatial_gamma"],
    n_components=best_kernel_params["spatial_n_components"],
    random_state=123,
)
Z_train_spat = spatial_map.fit_transform(X_train_spat)
Z_test_spat  = spatial_map.transform(X_test_spat)

# Concatenate into Z
Z_train = np.hstack([Z_train_temp, Z_train_spat, Z_train_other])
Z_test  = np.hstack([Z_test_temp,  Z_test_spat,  Z_test_other])

n_temp  = Z_train_temp.shape[1]
n_spat  = Z_train_spat.shape[1]
n_other = Z_train_other.shape[1]

block_slices = {
    "Temporal RFF (STL lags)": slice(0, n_temp),
    "Spatial RFF (coords/area/CoL dist)": slice(n_temp, n_temp + n_spat),
    "Other linear (socio/region/embeds)": slice(n_temp + n_spat, n_temp + n_spat + n_other),
}

# Scale Z (train only) — matches tuning setup for SGD stability
Z_scaler = StandardScaler()
Z_train = Z_scaler.fit_transform(Z_train)
Z_test  = Z_scaler.transform(Z_test)




## Final fit

In [8]:
model = SGDRegressor(
    loss="epsilon_insensitive",
    penalty="l2",
    epsilon=best_sgd_params["epsilon"],
    alpha=best_sgd_params["alpha"],
    learning_rate=best_sgd_params.get("learning_rate", "invscaling"),
    eta0=best_sgd_params.get("eta0", 0.01),
    max_iter=best_sgd_params.get("max_iter", 3000),
    tol=best_sgd_params.get("tol", 1e-3),
    random_state=best_sgd_params.get("random_state", 42),
)
model.fit(Z_train, y_train_scaled)

y_pred_scaled = model.predict(Z_test)
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

df_test = df_test.copy()
df_test["y_pred"] = y_pred
df_test["resid"] = y_test_true - y_pred

## Evaluation

In [9]:
# =========================================================
# Uncertainty (approximate)
# With SGD we don't have a closed-form posterior like Ridge.
# We'll use a simple homoskedastic Gaussian approximation:
# sigma_hat from TRAIN residuals in original £ scale.
# =========================================================
train_pred_scaled = model.predict(Z_train)
train_pred = y_scaler.inverse_transform(train_pred_scaled.reshape(-1, 1)).ravel()
train_true = y_train_raw.ravel()

sigma_hat = float(np.std(train_true - train_pred, ddof=1))
y_std = np.full_like(y_pred, sigma_hat, dtype=float)

z = 1.96
y_lower = y_pred - z * y_std
y_upper = y_pred + z * y_std

PICP = float(np.mean((y_test_true >= y_lower) & (y_test_true <= y_upper)))
PIW  = float(np.mean(y_upper - y_lower))
CRPS = crps_gaussian(y_test_true, y_pred, y_std)

# =========================================================
# Global metrics
# =========================================================
global_mae   = mae(y_test_true, y_pred)
global_rmse  = rmse(y_test_true, y_pred)
global_smape = smape(y_test_true, y_pred)
global_mase  = mase(y_test_true, y_pred, train_true, m=12)

print("=== Global accuracy (ARX Ridge) ===")
print(f"MAE   : {global_mae:,.3f}")
print(f"RMSE  : {global_rmse:,.3f}")
print(f"sMAPE : {global_smape:,.3f}%")
print(f"MASE  : {global_mase:,.3f}")

# =========================================================
# Across-LA consistency
# =========================================================
la_mae = df_test.groupby(ENTITY_COL).apply(lambda g: mae(g[TARGET_COL].values, g["y_pred"].values))
median_mae = float(la_mae.median())
p75_mae    = float(la_mae.quantile(0.75))

print("\n=== Across-LA consistency ===")
print(f"Median LA MAE       : {median_mae:,.3f}")
print(f"75th percentile MAE : {p75_mae:,.3f}")

# =========================================================
# Spatio-temporal diagnostics
# =========================================================
la_resid_mean = df_test.groupby(ENTITY_COL)["resid"].mean()

centroids = (
    df_test
    .dropna(subset=["centroid_x", "centroid_y"])
    .sort_values(TIME_COL)
    .groupby(ENTITY_COL)
    .tail(1)
    .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
)

# align
common = la_resid_mean.index.intersection(centroids.index)
la_resid_mean = la_resid_mean.loc[common]
centroids = centroids.loc[common]

mask_valid = centroids[["centroid_x", "centroid_y"]].notna().all(axis=1)
centroids_valid = centroids.loc[mask_valid]
la_resid_mean_valid = la_resid_mean.loc[centroids_valid.index]

print(f"\nLAs used for Moran's I: {len(centroids_valid)} / {len(la_resid_mean)}")

if len(centroids_valid) <= 1:
    I_moran = {"I": np.nan, "z_score": np.nan, "p_value": np.nan}
else:
    k_eff = min(5, len(centroids_valid) - 1)
    I_moran = morans_i(
        residuals=la_resid_mean_valid.values,
        xs=centroids_valid["centroid_x"].values,
        ys=centroids_valid["centroid_y"].values,
        k=k_eff,
        permutations=999,
        random_state=42,
    )
    print("\n=== Spatio-temporal diagnostics ===")
    print(f"Moran's I (mean residuals across LAs): {I_moran['I']:.4f}")
    print(f"Moran's I z score: {I_moran['z_score']:.4f}")
    print(f"Moran's I p value: {I_moran['p_value']:.4f}")

# Ljung–Box on monthly mean residuals (aggregate across LAs)
monthly_resid = df_test.groupby(TIME_COL)["resid"].mean().sort_index()
lb_res = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
q_stat = float(lb_res["lb_stat"].iloc[0])
p_val  = float(lb_res["lb_pvalue"].iloc[0])
print(f"Ljung–Box Q(12): stat={q_stat:.3f}, p={p_val:.4f}")

# =========================================================
# Direction & growth
# =========================================================
dir_acc = directional_accuracy(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred")
gre_mae = growth_rate_error(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", m=12)

print("\n=== Direction & growth ===")
print(f"Directional accuracy (MoM sign)   : {dir_acc:.3f}")
print(f"Growth-rate error MAE (12-month)  : {gre_mae:.4f}")


=== Global accuracy (ARX Ridge) ===
MAE   : 53,163.435
RMSE  : 63,235.724
sMAPE : 23.355%
MASE  : 2.526

=== Across-LA consistency ===
Median LA MAE       : 51,264.663
75th percentile MAE : 60,538.978

LAs used for Moran's I: 294 / 294

=== Spatio-temporal diagnostics ===
Moran's I (mean residuals across LAs): 0.2342
Moran's I z score: 6.4592
Moran's I p value: 0.0010
Ljung–Box Q(12): stat=81.088, p=0.0000

=== Direction & growth ===
Directional accuracy (MoM sign)   : 0.506
Growth-rate error MAE (12-month)  : 0.1738


C:\Users\slong\AppData\Local\Temp\ipykernel_16840\486894229.py:39: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  la_mae = df_test.groupby(ENTITY_COL).apply(lambda g: mae(g[TARGET_COL].values, g["y_pred"].values))


## Save results 

In [10]:
output_path = "../../results/final_SVR_results.xlsx"

summary_df = pd.DataFrame([{
    "model": "SGD_EPS_INSENSITIVE_BLOCKWISE_RFF",
    "train_end": str(TRAIN_END_DATE.date()),
    "test_start": str(TEST_START_DATE.date()),
    "lag_set": str(best_lag_set),
    "kernel_params": str(best_kernel_params),
    "sgd_params": str(best_sgd_params),
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,
    "Morans_I": float(I_moran["I"]) if isinstance(I_moran, dict) else np.nan,
    "Morans_I_z": float(I_moran["z_score"]) if isinstance(I_moran, dict) and I_moran.get("z_score") is not None else np.nan,
    "Morans_I_p": float(I_moran["p_value"]) if isinstance(I_moran, dict) and I_moran.get("p_value") is not None else np.nan,
    "LjungBox_Q12": q_stat,
    "LjungBox_p": p_val,
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae,
    "PICP_95": PICP,
    "PIW_95": PIW,
    "CRPS": CRPS,
    "Sigma_hat_train": sigma_hat,
}])

la_mae_df = la_mae.reset_index()
la_mae_df.columns = [ENTITY_COL, "LA_MAE"]

with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    summary_df.to_excel(writer, sheet_name="Global_Summary", index=False)
    la_mae_df.to_excel(writer, sheet_name="LA_MAE", index=False)
    df_test[[ENTITY_COL, TIME_COL, TARGET_COL, "y_pred", "resid"]].to_excel(
        writer, sheet_name="Test_Predictions", index=False
    )

print(f"\nResults saved to: {output_path}")


Results saved to: ../../results/final_SVR_results.xlsx


In [11]:
print("TEST rows:", len(df_test))
print("TEST date range:", df_test[TIME_COL].min(), "->", df_test[TIME_COL].max())
print("Unique months:", df_test[TIME_COL].nunique())
print("Unique LAs:", df_test[ENTITY_COL].nunique())

TEST rows: 7056
TEST date range: 2022-04-01 00:00:00 -> 2024-03-01 00:00:00
Unique months: 24
Unique LAs: 294


In [ ]:
z_feature_names = list(Z_train.columns) if hasattr(Z_train, "columns") else [f"z{i}" for i in range(Z_train.shape[1])]
coef = pd.Series(model.coef_.ravel(), index=z_feature_names)
coef_sorted = coef.reindex(coef.abs().sort_values(ascending=False).index)
print(coef_sorted.head(20))



z452    0.528193
z450    0.141042
z397    0.122935
z90     0.110804
z455    0.090806
z370   -0.087977
z80     0.084558
z465   -0.077932
z99    -0.071694
z462    0.070713
z30    -0.061011
z345   -0.057325
z377    0.055095
z384    0.052640
z466    0.048370
z428   -0.046982
z114   -0.046077
z143   -0.045631
z95    -0.045386
z109   -0.045262
dtype: float64


In [12]:
from sklearn.metrics import mean_absolute_error

def block_permutation_importance_mae(model, Z, y_true_scaled, y_scaler, block_slices, n_repeats=20, seed=42):
    """
    Computes permutation importance by shuffling whole blocks of Z together.
    Uses MAE in ORIGINAL units (inverse-transformed).
    """
    rng = np.random.default_rng(seed)

    # Baseline predictions in original units
    base_pred_scaled = model.predict(Z)
    base_pred = y_scaler.inverse_transform(base_pred_scaled.reshape(-1, 1)).ravel()
    y_true = y_scaler.inverse_transform(y_true_scaled.reshape(-1, 1)).ravel()
    base_mae = mean_absolute_error(y_true, base_pred)

    out = []
    Zp = Z.copy()

    for name, sl in block_slices.items():
        maes = []
        for _ in range(n_repeats):
            # shuffle rows within the block columns
            idx = rng.permutation(Zp.shape[0])
            saved = Zp[:, sl].copy()
            Zp[:, sl] = Zp[idx, sl]
            pred_scaled = model.predict(Zp)
            pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
            maes.append(mean_absolute_error(y_true, pred))
            Zp[:, sl] = saved

        imp = np.mean(maes) - base_mae
        out.append((name, imp, np.std(maes)))

    df_out = pd.DataFrame(out, columns=["block", "perm_imp_mae", "std_mae"]).sort_values("perm_imp_mae", ascending=False)
    return df_out, base_mae

In [13]:
y_test_scaled = y_scaler.transform(df_test[TARGET_COL].values.reshape(-1, 1)).ravel()

block_imp, base_mae = block_permutation_importance_mae(
    model=model,
    Z=Z_test,
    y_true_scaled=y_test_scaled,
    y_scaler=y_scaler,
    block_slices=block_slices,
    n_repeats=20,
    seed=42
)

print("Baseline MAE:", base_mae)
print(block_imp)

Baseline MAE: 53163.43505046387
                                block  perm_imp_mae     std_mae
2  Other linear (socio/region/embeds)  70991.014935  739.772077
0             Temporal RFF (STL lags)  29579.491254  396.807342
1  Spatial RFF (coords/area/CoL dist)   3747.160713  124.529379


In [ ]:
# block_imp: DataFrame with columns ["block", "perm_imp_mae", "std_mae"]
# base_mae: float

block_imp_out = block_imp.copy()
block_imp_out["baseline_mae"] = base_mae  # same value repeated for convenience

out_path = "../../results/svm_block_importance.csv"
block_imp_out.to_csv(out_path, index=False)
print("Saved:", out_path)
